In [1]:
from datetime import datetime
from typing import Any, Callable


class GenericPriorityQueue:
    """A generic priority queue that uses a custom comparator function."""

    def __init__(self, comparator: Callable[[Any, Any], int]) -> None:
        """Initialize the queue with a custom comparator function.

        Args:
            comparator: A function that takes two items and returns a negative
                number if the first has higher priority, a positive number if
                the second has higher priority, and zero if they are equal.

        Raises:
            TypeError: If comparator is not callable.
        """
        if not callable(comparator):
            raise TypeError("Comparator must be a callable function.")
        self._comparator = comparator
        self._heap: list[Any] = []

    def _parent(self, index: int) -> int:
        return (index - 1) // 2

    def _left_child(self, index: int) -> int:
        return 2 * index + 1

    def _right_child(self, index: int) -> int:
        return 2 * index + 2

    def _sift_up(self, index: int) -> None:
        while index > 0:
            parent = self._parent(index)
            cmp = self._compare(self._heap[index], self._heap[parent])
            if cmp < 0:
                self._heap[index], self._heap[parent] = self._heap[parent], self._heap[index]
                index = parent
            else:
                break

    def _sift_down(self, index: int) -> None:
        size = len(self._heap)
        while True:
            largest = index
            left = self._left_child(index)
            right = self._right_child(index)

            if left < size:
                cmp = self._compare(self._heap[left], self._heap[largest])
                if cmp < 0:
                    largest = left

            if right < size:
                cmp = self._compare(self._heap[right], self._heap[largest])
                if cmp < 0:
                    largest = right

            if largest != index:
                self._heap[index], self._heap[largest] = self._heap[largest], self._heap[index]
                index = largest
            else:
                break

    def _compare(self, a: Any, b: Any) -> int:
        """Use the comparator to compare two items.

        Returns a negative number if a has higher priority, positive if b has
        higher priority, and zero if they are equal.
        """
        return self._comparator(a, b)

    def enqueue(self, item: Any) -> None:
        """Add an item to the queue.

        Args:
            item: The item to add.

        Raises:
            TypeError: If the item is incompatible with existing items.
        """
        if self._heap:
            try:
                self._compare(self._heap[0], item)
            except TypeError:
                raise TypeError(
                    f"Item of type {type(item).__name__} is incompatible "
                    f"with items in the queue."
                )
        self._heap.append(item)
        self._sift_up(len(self._heap) - 1)

    def dequeue(self) -> Any:
        """Remove and return the highest priority item.

        Returns:
            The highest priority item.

        Raises:
            IndexError: If the queue is empty.
        """
        if self.is_empty():
            raise IndexError("Dequeue from an empty queue.")
        if len(self._heap) == 1:
            return self._heap.pop()
        top = self._heap[0]
        self._heap[0] = self._heap.pop()
        if not self.is_empty():
            self._sift_down(0)
        return top

    def peek(self) -> Any:
        """View the highest priority item without removing it.

        Returns:
            The highest priority item.

        Raises:
            IndexError: If the queue is empty.
        """
        if self.is_empty():
            raise IndexError("Peek from an empty queue.")
        return self._heap[0]

    def is_empty(self) -> bool:
        """Check if the queue is empty.

        Returns:
            True if the queue is empty, False otherwise.
        """
        return len(self._heap) == 0

    def size(self) -> int:
        """Return the number of items in the queue.

        Returns:
            The number of items.
        """
        return len(self._heap)


class Task:
    """A task with priority, deadline, and description."""

    def __init__(self, priority: int, deadline: datetime, description: str) -> None:
        self.priority = priority
        self.deadline = deadline
        self.description = description

    def __repr__(self) -> str:
        return (f"Task(priority={self.priority}, deadline={self.deadline}, "
                f"description={self.description!r})")

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Task):
            return NotImplemented
        return (self.priority == other.priority
                and self.deadline == other.deadline
                and self.description == other.description)


class Patient:
    """A patient with severity, wait time, and name."""

    def __init__(self, severity: int, wait_time: int, name: str) -> None:
        self.severity = severity
        self.wait_time = wait_time
        self.name = name

    def __repr__(self) -> str:
        return (f"Patient(severity={self.severity}, wait_time={self.wait_time}, "
                f"name={self.name!r})")

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Patient):
            return NotImplemented
        return (self.severity == other.severity
                and self.wait_time == other.wait_time
                and self.name == other.name)



def task_comparator(a: Task, b: Task) -> int:
    """Compare two tasks.

    Order by priority (higher first), then by deadline (earlier first).
    """
    if a.priority != b.priority:
        return b.priority - a.priority
    if a.deadline < b.deadline:
        return -1
    elif a.deadline > b.deadline:
        return 1
    return 0


def patient_comparator(a: Patient, b: Patient) -> int:
    """Compare two patients.

    Order by severity (higher first), then by wait time (longer first).
    """
    if a.severity != b.severity:
        return b.severity - a.severity
    return b.wait_time - a.wait_time


def _test_basic_operations() -> None:
    """Test basic enqueue, dequeue, peek, is_empty, size operations."""
    print("=== Basic Operations ===")

    def int_comparator(a: int, b: int) -> int:
        return b - a

    pq = GenericPriorityQueue(int_comparator)

    assert pq.is_empty(), "Queue should be empty initially"
    assert pq.size() == 0, "Size should be 0 initially"

    pq.enqueue(10)
    pq.enqueue(30)
    pq.enqueue(20)
    assert pq.size() == 3, "Size should be 3 after 3 enqueues"
    assert not pq.is_empty(), "Queue should not be empty"

    assert pq.peek() == 30, "Peek should return 30"

    assert pq.dequeue() == 30, "First dequeue should return 30"
    assert pq.dequeue() == 20, "Second dequeue should return 20"
    assert pq.dequeue() == 10, "Third dequeue should return 10"
    assert pq.is_empty(), "Queue should be empty after all dequeues"
    assert pq.size() == 0, "Size should be 0"

    try:
        pq.peek()
        assert False, "Should have raised IndexError"
    except IndexError:
        pass

    try:
        pq.dequeue()
        assert False, "Should have raised IndexError"
    except IndexError:
        pass

    print("  PASSED: Basic operations")


def _test_different_comparators() -> None:
    """Test priority queues with different comparator functions."""
    print("=== Different Comparators ===")

    def min_comparator(a: int, b: int) -> int:
        return a - b

    pq = GenericPriorityQueue(min_comparator)
    pq.enqueue(5)
    pq.enqueue(1)
    pq.enqueue(3)
    assert pq.dequeue() == 1, "Min comparator: should dequeue 1 first"
    assert pq.dequeue() == 3, "Min comparator: should dequeue 3 second"
    assert pq.dequeue() == 5, "Min comparator: should dequeue 5 last"

    def str_comparator(a: str, b: str) -> int:
        if a < b:
            return -1
        elif a > b:
            return 1
        return 0

    pq2 = GenericPriorityQueue(str_comparator)
    pq2.enqueue("banana")
    pq2.enqueue("apple")
    pq2.enqueue("cherry")
    assert pq2.dequeue() == "apple", "String comparator: should dequeue apple first"
    assert pq2.dequeue() == "banana", "String comparator: should dequeue banana second"
    assert pq2.dequeue() == "cherry", "String comparator: should dequeue cherry last"

    print("  PASSED: Different comparators")


def _test_task_data_type() -> None:
    """Test Task objects with task_comparator."""
    print("=== Task Data Type ===")

    pq = GenericPriorityQueue(task_comparator)

    t1 = Task(1, datetime(2026, 11, 1), "Low priority task")
    t2 = Task(3, datetime(2026, 10, 30), "High priority task")
    t3 = Task(2, datetime(2026, 11, 5), "Medium priority task")
    t4 = Task(3, datetime(2026, 10, 28), "Another high priority task")

    pq.enqueue(t1)
    pq.enqueue(t2)
    pq.enqueue(t3)
    pq.enqueue(t4)

    first = pq.dequeue()
    assert first.priority == 3, f"First should have priority 3, got {first.priority}"
    assert first.deadline == datetime(2026, 10, 28), \
        f"First should have deadline 2026-10-28, got {first.deadline}"

    second = pq.dequeue()
    assert second.priority == 3, f"Second should have priority 3, got {second.priority}"
    assert second.deadline == datetime(2026, 10, 30), \
        f"Second should have deadline 2026-10-30, got {second.deadline}"

    third = pq.dequeue()
    assert third.priority == 2, f"Third should have priority 2, got {third.priority}"

    fourth = pq.dequeue()
    assert fourth.priority == 1, f"Fourth should have priority 1, got {fourth.priority}"

    assert pq.is_empty(), "Queue should be empty"
    print("  PASSED: Task data type")


def _test_patient_data_type() -> None:
    """Test Patient objects with patient_comparator."""
    print("=== Patient Data Type ===")

    pq = GenericPriorityQueue(patient_comparator)

    p1 = Patient(1, 60, "Alice")
    p2 = Patient(5, 30, "Bob")
    p3 = Patient(3, 120, "Charlie")
    p4 = Patient(5, 45, "Diana")

    pq.enqueue(p1)
    pq.enqueue(p2)
    pq.enqueue(p3)
    pq.enqueue(p4)

    first = pq.dequeue()
    assert first.severity == 5, f"First should have severity 5, got {first.severity}"
    assert first.wait_time == 45, \
        f"First should have wait_time 45, got {first.wait_time}"
    assert first.name == "Diana", f"First should be Diana, got {first.name}"

    second = pq.dequeue()
    assert second.severity == 5, f"Second should have severity 5, got {second.severity}"
    assert second.wait_time == 30, \
        f"Second should have wait_time 30, got {second.wait_time}"
    assert second.name == "Bob", f"Second should be Bob, got {second.name}"

    third = pq.dequeue()
    assert third.severity == 3, f"Third should have severity 3, got {third.severity}"
    assert third.name == "Charlie", f"Third should be Charlie, got {third.name}"

    fourth = pq.dequeue()
    assert fourth.severity == 1, f"Fourth should have severity 1, got {fourth.severity}"
    assert fourth.name == "Alice", f"Fourth should be Alice, got {fourth.name}"

    assert pq.is_empty(), "Queue should be empty"
    print("  PASSED: Patient data type")


def _test_mixed_data_types_error() -> None:
    """Test that mixing incompatible data types raises TypeError."""
    print("=== Mixed Data Types Error ===")

    pq = GenericPriorityQueue(task_comparator)
    pq.enqueue(Task(1, datetime(2026, 11, 1), "Task 1"))

    try:
        pq.enqueue(Patient(5, 10, "Bob"))
        assert False, "Should have raised TypeError"
    except TypeError:
        pass

    pq2 = GenericPriorityQueue(patient_comparator)
    pq2.enqueue(Patient(5, 10, "Bob"))

    try:
        pq2.enqueue(Task(1, datetime(2026, 11, 1), "Task 1"))
        assert False, "Should have raised TypeError"
    except TypeError:
        pass

    print("  PASSED: Mixed data types error")


def _test_invalid_comparator() -> None:
    """Test that passing a non-callable as comparator raises TypeError."""
    print("=== Invalid Comparator Error ===")

    try:
        GenericPriorityQueue("not a function")
        assert False, "Should have raised TypeError"
    except TypeError as e:
        assert "callable" in str(e).lower(), \
            f"Error message should mention 'callable', got: {e}"

    try:
        GenericPriorityQueue(42)
        assert False, "Should have raised TypeError"
    except TypeError:
        pass

    try:
        GenericPriorityQueue(None)
        assert False, "Should have raised TypeError"
    except TypeError:
        pass

    print("  PASSED: Invalid comparator error")


def _test_empty_queue_operations() -> None:
    """Test operations on an empty queue."""
    print("=== Empty Queue Operations ===")

    pq = GenericPriorityQueue(lambda a, b: 0)
    assert pq.is_empty(), "Queue should be empty"
    assert pq.size() == 0, "Size should be 0"

    try:
        pq.dequeue()
        assert False, "Should have raised IndexError"
    except IndexError:
        pass

    try:
        pq.peek()
        assert False, "Should have raised IndexError"
    except IndexError:
        pass

    print("  PASSED: Empty queue operations")


def _test_single_item() -> None:
    """Test queue with a single item."""
    print("=== Single Item Operations ===")

    pq = GenericPriorityQueue(lambda a, b: b - a)
    pq.enqueue(42)

    assert not pq.is_empty(), "Queue should not be empty"
    assert pq.size() == 1, "Size should be 1"
    assert pq.peek() == 42, "Peek should return 42"
    assert pq.dequeue() == 42, "Dequeue should return 42"
    assert pq.is_empty(), "Queue should be empty after dequeue"
    assert pq.size() == 0, "Size should be 0"

    print("  PASSED: Single item operations")


def _test_same_priority_items() -> None:
    """Test multiple items with the same priority."""
    print("=== Same Priority Items ===")

    pq = GenericPriorityQueue(task_comparator)

    t1 = Task(1, datetime(2026, 12, 1), "Task A")
    t2 = Task(1, datetime(2026, 11, 15), "Task B")
    t3 = Task(1, datetime(2026, 11, 1), "Task C")

    pq.enqueue(t1)
    pq.enqueue(t2)
    pq.enqueue(t3)

    first = pq.dequeue()
    assert first.deadline == datetime(2026, 11, 1), \
        f"First should have deadline 2026-11-01, got {first.deadline}"
    assert first.description == "Task C"

    second = pq.dequeue()
    assert second.deadline == datetime(2026, 11, 15), \
        f"Second should have deadline 2026-11-15, got {second.deadline}"
    assert second.description == "Task B"

    third = pq.dequeue()
    assert third.deadline == datetime(2026, 12, 1), \
        f"Third should have deadline 2026-12-01, got {third.deadline}"
    assert third.description == "Task A"

    pq2 = GenericPriorityQueue(patient_comparator)
    p1 = Patient(3, 10, "Alice")
    p2 = Patient(3, 30, "Bob")
    p3 = Patient(3, 20, "Charlie")

    pq2.enqueue(p1)
    pq2.enqueue(p2)
    pq2.enqueue(p3)

    first = pq2.dequeue()
    assert first.name == "Bob", f"First should be Bob, got {first.name}"
    assert first.wait_time == 30

    second = pq2.dequeue()
    assert second.name == "Charlie", f"Second should be Charlie, got {second.name}"
    assert second.wait_time == 20

    third = pq2.dequeue()
    assert third.name == "Alice", f"Third should be Alice, got {third.name}"
    assert third.wait_time == 10

    print("  PASSED: Same priority items")


def _test_large_number_of_items() -> None:
    """Test with a large number of items for performance and correctness."""
    print("=== Large Number of Items ===")

    pq = GenericPriorityQueue(lambda a, b: b - a)

    import random
    items = list(range(1000))
    random.shuffle(items)

    for item in items:
        pq.enqueue(item)

    assert pq.size() == 1000, "Size should be 1000"

    expected = 999
    for _ in range(1000):
        item = pq.dequeue()
        assert item == expected, f"Expected {expected}, got {item}"
        expected -= 1

    assert pq.is_empty(), "Queue should be empty"
    assert pq.size() == 0, "Size should be 0"

    print("  PASSED: Large number of items")


def _test_peek_does_not_remove() -> None:
    """Test that peek does not remove the item."""
    print("=== Peek Does Not Remove ===")

    pq = GenericPriorityQueue(lambda a, b: b - a)
    pq.enqueue(100)
    pq.enqueue(200)
    pq.enqueue(300)

    assert pq.peek() == 300
    assert pq.peek() == 300
    assert pq.peek() == 300

    assert pq.size() == 3, "Size should still be 3"

    assert pq.dequeue() == 300
    assert pq.size() == 2

    print("  PASSED: Peek does not remove")


def _test_enqueue_preserves_all_items() -> None:
    """Test that enqueue preserves all items correctly."""
    print("=== Enqueue Preserves All Items ===")

    pq = GenericPriorityQueue(lambda a, b: b - a)

    for i in range(1, 101):
        pq.enqueue(i)

    assert pq.size() == 100, "Size should be 100"

    for expected in range(100, 0, -1):
        item = pq.dequeue()
        assert item == expected, f"Expected {expected}, got {item}"

    assert pq.is_empty(), "Queue should be empty"
    print("  PASSED: Enqueue preserves all items")


def _test_repeated_enqueue_dequeue() -> None:
    """Test interleaved enqueue and dequeue operations."""
    print("=== Repeated Enqueue/Dequeue ===")

    pq = GenericPriorityQueue(lambda a, b: b - a)

    pq.enqueue(5)
    assert pq.dequeue() == 5

    pq.enqueue(10)
    pq.enqueue(3)
    assert pq.dequeue() == 10
    assert pq.dequeue() == 3

    pq.enqueue(7)
    pq.enqueue(15)
    pq.enqueue(1)
    assert pq.dequeue() == 15
    assert pq.dequeue() == 7
    assert pq.dequeue() == 1

    assert pq.is_empty(), "Queue should be empty"
    print("  PASSED: Repeated enqueue/dequeue")


def run_all_tests() -> None:
    """Run all test scenarios."""
    print("Running GenericPriorityQueue Tests\n")
    print("=" * 50)

    tests = [
        _test_basic_operations,
        _test_different_comparators,
        _test_task_data_type,
        _test_patient_data_type,
        _test_mixed_data_types_error,
        _test_invalid_comparator,
        _test_empty_queue_operations,
        _test_single_item,
        _test_same_priority_items,
        _test_large_number_of_items,
        _test_peek_does_not_remove,
        _test_enqueue_preserves_all_items,
        _test_repeated_enqueue_dequeue,
    ]

    passed = 0
    failed = 0

    for test_fn in tests:
        try:
            test_fn()
            passed += 1
        except AssertionError as e:
            print(f"  FAILED: {test_fn.__doc__.strip()}")
            print(f"    Error: {e}")
            failed += 1
        except Exception as e:
            print(f"  ERROR: {test_fn.__doc__.strip()}")
            print(f"    {type(e).__name__}: {e}")
            failed += 1

    print("=" * 50)
    print(f"\nResults: {passed} passed, {failed} failed out of {len(tests)} tests")


if __name__ == "__main__":
    run_all_tests()


Running GenericPriorityQueue Tests

=== Basic Operations ===
  PASSED: Basic operations
=== Different Comparators ===
  PASSED: Different comparators
=== Task Data Type ===
  PASSED: Task data type
=== Patient Data Type ===
  PASSED: Patient data type
=== Mixed Data Types Error ===
  ERROR: Test that mixing incompatible data types raises TypeError.
    AttributeError: 'Patient' object has no attribute 'priority'
=== Invalid Comparator Error ===
  PASSED: Invalid comparator error
=== Empty Queue Operations ===
  PASSED: Empty queue operations
=== Single Item Operations ===
  PASSED: Single item operations
=== Same Priority Items ===
  PASSED: Same priority items
=== Large Number of Items ===
  PASSED: Large number of items
=== Peek Does Not Remove ===
  PASSED: Peek does not remove
=== Enqueue Preserves All Items ===
  PASSED: Enqueue preserves all items
=== Repeated Enqueue/Dequeue ===
  PASSED: Repeated enqueue/dequeue

Results: 12 passed, 1 failed out of 13 tests


In [2]:
# Test cases
def test_generic_priority_queue():
    """Comprehensive test cases for the GenericPriorityQueue implementation."""
    
    # Test 1: Basic integer priority queue
    print("Test 1: Basic integer operations")
    int_queue = GenericPriorityQueue(lambda a, b: a - b)  # Min heap
    numbers = [5, 2, 8, 1, 9, 3]
    for num in numbers:
        int_queue.enqueue(num)
    
    assert int_queue.peek() == 1, "Minimum element should be 1"
    assert int_queue.dequeue() == 1, "Should dequeue 1"
    assert int_queue.size() == 5, "Size should be 5"
    print("Basic integer operations passed")
 
    # Test 2: Task priority queue
    print("\nTest 2: Task scheduling")
    task_queue = GenericPriorityQueue(task_comparator)
    now = datetime.now()
    tasks = [
        Task(1, now + timedelta(hours=1), "Low priority task"),
        Task(3, now + timedelta(hours=2), "High priority task"),
        Task(2, now + timedelta(hours=1), "Medium priority task"),
        Task(3, now + timedelta(hours=1), "High priority task, earlier deadline")
    ]
    
    for task in tasks:
        task_queue.enqueue(task)
    
    highest_priority_task = task_queue.dequeue()
    assert highest_priority_task.priority == 3
    assert highest_priority_task.deadline == now + timedelta(hours=1)
    print("Task scheduling tests passed")
 
    # Test 3: Patient priority queue
    print("\nTest 3: Patient triage")
    patient_queue = GenericPriorityQueue(patient_comparator)
    patients = [
        Patient(2, 10, "John"),
        Patient(3, 5, "Alice"),
        Patient(3, 10, "Bob"),
        Patient(1, 20, "Charlie")
    ]
    
    for patient in patients:
        patient_queue.enqueue(patient)
    
    highest_priority_patient = patient_queue.dequeue()
    assert highest_priority_patient.severity == 3
    assert highest_priority_patient.wait_time == 10
    print("Patient triage tests passed")
 
    # Test 4: Error handling
    print("\nTest 4: Error handling")
    try:
        int_queue.enqueue("not an integer")
        assert False, "Should raise TypeError"
    except TypeError:
        print("Type error handling passed")
 
    try:
        empty_queue = GenericPriorityQueue(lambda a, b: a - b)
        empty_queue.dequeue()
        assert False, "Should raise IndexError"
    except IndexError:
        print("Empty queue error handling passed")
 
    # Test 5: Edge cases
    print("\nTest 5: Edge cases")
    edge_queue = GenericPriorityQueue(lambda a, b: a - b)
    
    # Single element
    edge_queue.enqueue(1)
    assert edge_queue.dequeue() == 1
    assert edge_queue.is_empty()
    
    # Multiple elements with same priority
    same_priority = [5, 5, 5, 5]
    for num in same_priority:
        edge_queue.enqueue(num)
    
    assert all(edge_queue.dequeue() == 5 for _ in range(4))
    print("Edge cases passed")
 
if __name__ == "__main__":
    test_generic_priority_queue()

Test 1: Basic integer operations
Basic integer operations passed

Test 2: Task scheduling


NameError: name 'timedelta' is not defined